# Set up

In [13]:
!pip install scikit-survival

In [14]:
import pandas as pd
import os
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
import xgboost as xgb
import shap
import random
from sklearn.feature_selection import VarianceThreshold, RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import Ridge
def seed_everything(seed=42):
    """
    Hàm cố định toàn bộ các yếu tố ngẫu nhiên trong môi trường Python, 
    NumPy và các thư viện Học máy để đảm bảo kết quả luôn giống hệt nhau ở mỗi lần chạy.
    """
    # 1. Cố định random của Python cơ bản
    random.seed(seed)
    
    # 2. Cố định biến môi trường Hash của Python (ảnh hưởng đến dictionary, set)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. Cố định random của NumPy (CỰC KỲ QUAN TRỌNG cho hàm apply_psdata của bạn)
    np.random.seed(seed)
    
    print(f"Đã khóa toàn bộ seed với giá trị: {seed}")

# Gọi hàm ngay lập tức
seed_everything(seed=42)

Đã khóa toàn bộ seed với giá trị: 42


In [15]:
# 1. Tải dữ liệu
DATA_DIR = Path("/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26")
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")

# 2. Chuẩn bị Target (y)
y_train = np.empty(len(train), dtype=[('event', bool), ('time', float)])
y_train['event'] = train['event'].astype(bool)
y_train['time'] = train['time_to_hit_hours']

# 3. Chuẩn bị Features (X)
features_to_drop = ['event_id', 'time_to_hit_hours', 'event']
X_train = train.drop(columns=features_to_drop)
X_test = test.drop(columns=['event_id'])

# Feature Selection

In [16]:
# --- BƯỚC 1: LỌC THÔ (FILTER METHODS) ---
def step1_raw_filtering(X, threshold_corr=0.9):
    print("--- Bước 1: Đang lọc thô ---")
    selector = VarianceThreshold(threshold=0.01)
    X_high_v = pd.DataFrame(selector.fit_transform(X), columns=X.columns[selector.get_support()])
    
    corr_matrix = X_high_v.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold_corr)]
    X_filtered = X_high_v.drop(columns=to_drop)
    
    print(f"Đã loại bỏ {len(X.columns) - len(X_filtered.columns)} features.")
    return X_filtered

# --- BƯỚC 2: EMBEDDED METHODS (XGBOOST SURVIVAL IMPORTANCE) ---
def step2_xgb_importance(X, y_survival):
    print("\n--- Bước 2: Tính toán tầm quan trọng bằng XGBoost Survival ---")
    dtrain = xgb.DMatrix(X, label=y_survival)
    
    params = {
        'objective': 'survival:cox',
        'tree_method': 'hist', 
        'learning_rate': 0.05,
        'max_depth': 5
    }
    
    model = xgb.train(params, dtrain, num_boost_round=100)
    importance = model.get_score(importance_type='gain')
    importance_df = pd.DataFrame(importance.items(), columns=['Feature', 'Gain']).sort_values(by='Gain', ascending=False)
    
    return importance_df

# --- BƯỚC 3: WRAPPER METHODS (RFE) ---
def step3_rfe_refining(X, y_survival, top_n=30):
    print(f"\n--- Bước 3: Đang tinh lọc từ {top_n} features xuống 20 bằng RFE ---")
    estimator = xgb.XGBRegressor(objective='survival:cox', n_estimators=50)
    
    selector = RFE(estimator, n_features_to_select=20, step=2)
    selector = selector.fit(X, y_survival)
    
    selected_features = X.columns[selector.get_support()]
    return selected_features

# --- BƯỚC 4: KIỂM CHỨNG NÂNG CAO (SHAP VALUES) ---
def step4_shap_validation(X_selected, y_survival):
    print("\n--- Bước 4: Kiểm chứng bằng SHAP Values ---")
    model = xgb.XGBRegressor(objective='survival:cox').fit(X_selected, y_survival)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_selected)
    
    shap.summary_plot(shap_values, X_selected)
    return shap_values

# ==========================================
# THỰC THI TOÀN BỘ QUY TRÌNH VỚI BIẾN MỤC TIÊU MỚI
# ==========================================

# 1. Chuẩn bị nhãn Survival (Cox)
# Quy tắc: y > 0 nếu event=1, y < 0 nếu event=0 (censored)
y_survival = train['time_to_hit_hours'].copy()
y_survival[train['event'] == 0] = -y_survival[train['event'] == 0]

# 2. Loại bỏ các cột mục tiêu khỏi tập Feature nếu chúng còn nằm trong X_train
features_only = X_train.drop(columns=['time_to_hit_hours', 'event'], errors='ignore')

# Thực hiện quy trình
X_step1 = step1_raw_filtering(features_only)
importance_df = step2_xgb_importance(X_step1, y_survival)

# Lấy Top 30 feature từ bước 2
top_30_features = importance_df['Feature'].head(30).tolist()
final_feature_names = step3_rfe_refining(X_step1[top_30_features], y_survival)

print(f"\nDANH SÁCH FEATURE CUỐI CÙNG ({len(final_feature_names)}):")
print(final_feature_names)


--- Bước 1: Đang lọc thô ---
Đã loại bỏ 13 features.

--- Bước 2: Tính toán tầm quan trọng bằng XGBoost Survival ---

--- Bước 3: Đang tinh lọc từ 30 features xuống 20 bằng RFE ---

DANH SÁCH FEATURE CUỐI CÙNG (16):
Index(['dist_min_ci_0_5h', 'dt_first_last_0_5h', 'num_perimeters_0_5h',
       'dist_accel_m_per_h2', 'event_start_hour', 'spread_bearing_deg',
       'alignment_cos', 'event_start_dayofweek', 'event_start_month',
       'area_first_ha', 'area_growth_rel_0_5h', 'along_track_speed',
       'alignment_abs', 'area_growth_abs_0_5h', 'spread_bearing_sin',
       'cross_track_component'],
      dtype='object')


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_rfe.py:300: UserWarning: Found n_features_to_select=20 > n_features=16. There will be no feature selection and all features will be kept.
  warnings.warn(


# Augmentation

In [17]:
def apply_psdata(X, y, oversample_ratio=0.2, k=5, M=25):
    """
    Kỹ thuật PSDATA: Tăng cường dữ liệu thiểu số cho Survival Analysis
    - oversample_ratio: Tỷ lệ mẫu muốn tạo thêm so với tập gốc (VD: 0.2 = tạo thêm 20%)
    """
    num_samples = int(len(X) * oversample_ratio)
    print(f"\n--- THỰC THI PSDATA: Đang tổng hợp thêm {num_samples} mẫu mới ---")
    
    # Lọc các mẫu KHÔNG bị kiểm duyệt (event == True)
    uncensored_mask = y['event'] == True
    X_uncensored = X[uncensored_mask].values
    
    if len(X_uncensored) < k + 1:
        k = len(X_uncensored) - 1
        
    # Mô hình KNN cho Bước 1 (Tìm k láng giềng uncensored)
    nn_feature = NearestNeighbors(n_neighbors=k+1)
    nn_feature.fit(X_uncensored)
    
    # Mô hình KNN cho Bước 2 (Tìm M láng giềng trên toàn bộ dữ liệu)
    nn_all = NearestNeighbors(n_neighbors=M)
    nn_all.fit(X.values)
    
    new_X, new_T = [], []
    
    for _ in range(num_samples):
        # BƯỚC 1: Tổng hợp X_new dựa trên SMOTE
        idx = np.random.randint(0, len(X_uncensored))
        x_i = X_uncensored[idx]
        
        distances, indices = nn_feature.kneighbors([x_i])
        neighbor_idx = np.random.choice(indices[0][1:]) # Chọn 1 láng giềng ngẫu nhiên (bỏ chính nó)
        x_hat = X_uncensored[neighbor_idx]
        
        gap = np.random.uniform(0, 1, size=x_i.shape)
        x_new = x_i + gap * (x_hat - x_i) # Công thức (12)
        new_X.append(x_new)
        
        # BƯỚC 2: Tổng hợp T_new (Xấp xỉ Buckley-James bằng Linear Regression cục bộ)
        _, indices_M = nn_all.kneighbors([x_new])
        X_local = X.values[indices_M[0]]
        y_local_time = y['time'][indices_M[0]]
        
        # Fit mô hình cục bộ để tìm T_new = beta^T * X_new (Công thức 13)
        lr = lr = Ridge(alpha=0.5)
        lr.fit(X_local, y_local_time)
        t_new = lr.predict([x_new])[0]
        
        # Đảm bảo thời gian nằm trong khoảng logic (0 - 72h)
        t_new = np.clip(t_new, 0, 72)
        new_T.append(t_new)
        
    # Gộp dữ liệu mới vào dữ liệu cũ
    X_aug = pd.DataFrame(new_X, columns=X.columns)
    X_combined = pd.concat([X, X_aug], ignore_index=True)
    
    y_aug = np.empty(num_samples, dtype=[('event', bool), ('time', float)])
    y_aug['event'] = True  # Các mẫu PSDATA tạo ra được quy ước là uncensored
    y_aug['time'] = new_T
    y_combined = np.concatenate([y, y_aug])
    
    return X_combined, y_combined

# Training

In [18]:
# --- HÀM XỬ LÝ AN TOÀN KHI NGOẠI SUY THỜI GIAN ---
def safe_prob(fn, t):
    max_time = fn.domain[1]
    if t > max_time:
        return 1 - fn(max_time)
    return 1 - fn(t)

# --- HÀM TÍNH ĐIỂM ĐÁNH GIÁ CỦA CUỘC THI ---
def evaluate_wids_metric(y_true, risk_scores, surv_funcs):
    c_index = concordance_index_censored(y_true['event'], y_true['time'], risk_scores)[0]
    
    prob_12 = np.array([safe_prob(fn, 12) for fn in surv_funcs])
    prob_24 = np.array([safe_prob(fn, 24) for fn in surv_funcs])
    prob_48 = np.array([safe_prob(fn, 48) for fn in surv_funcs])
    prob_72 = np.array([safe_prob(fn, 72) for fn in surv_funcs])
    
    def brier_at_h(H, prob_h):
        mask_excluded = (~y_true['event']) & (y_true['time'] < H)
        mask_valid = ~mask_excluded
        
        y_valid = y_true[mask_valid]
        p_valid = prob_h[mask_valid]
        
        target = (y_valid['event'] & (y_valid['time'] <= H)).astype(float)
        return np.mean((target - p_valid)**2)

    brier_12 = brier_at_h(12, prob_12)
    brier_24 = brier_at_h(24, prob_24)
    brier_48 = brier_at_h(48, prob_48)
    brier_72 = brier_at_h(72, prob_72)
    
    wbs = 0.3 * brier_24 + 0.4 * brier_48 + 0.3 * brier_72
    hybrid_score = 0.3 * c_index + 0.7 * (1 - wbs)
    
    return hybrid_score, c_index, wbs, brier_12, brier_24, brier_48, brier_72


# --- SỬA LỖI 2: CHỈNH ĐÚNG BIẾN TARGET LÀ y_train ---
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# 4. ÁP DỤNG PSDATA (CHỈ TRÊN TẬP TRAIN)
X_tr_aug, y_tr_aug = apply_psdata(X_tr, y_tr, oversample_ratio=0.2)

# 5. Huấn luyện mô hình trên tập Train (80%) để kiểm tra Validation
print("Đang huấn luyện mô hình Gradient Boosting Survival (Tập Train 80%)...")
gbsa = GradientBoostingSurvivalAnalysis(
    n_estimators=100,      
    learning_rate=0.1,     
    max_depth=4,           
    subsample=0.8,         
    random_state=42
)
gbsa.fit(X_tr_aug, y_tr_aug)

# 6. Đánh giá mô hình trên tập Validation (20%)
print("Đang đánh giá trên tập validation (20%)...")
val_risk_scores = gbsa.predict(X_val)
val_surv_funcs = gbsa.predict_survival_function(X_val)

hybrid, c_idx, wbs, b12, b24, b48, b72 = evaluate_wids_metric(y_val, val_risk_scores, val_surv_funcs)

print("\n--- KẾT QUẢ LOCAL VALIDATION ---")
print(f"Hybrid Score    : {hybrid:.4f}")
print(f"C-index         : {c_idx:.4f}")
print(f"Weighted Brier  : {wbs:.4f}")
print(f"  - Brier@12h   : {b12:.4f}")
print(f"  - Brier@24h   : {b24:.4f}")
print(f"  - Brier@48h   : {b48:.4f}")
print(f"  - Brier@72h   : {b72:.4f}\n")

# --- SỬA LỖI 3: HUẤN LUYỆN TRÊN TẬP X_train VÀ DỰ ĐOÁN TRÊN X_test ---

X_train_aug, y_train_aug = apply_psdata(X_train, y_train, oversample_ratio=0.2)
print("Đang huấn luyện lại mô hình trên TOÀN BỘ dữ liệu train (100%)...")
gbsa.fit(X_train_aug, y_train_aug)

print("Đang dự đoán trên tập test...")
test_risk_scores = gbsa.predict(X_test)
test_surv_funcs = gbsa.predict_survival_function(X_test)

prob_12_test = np.array([safe_prob(fn, 12) for fn in test_surv_funcs])
prob_24_test = np.array([safe_prob(fn, 24) for fn in test_surv_funcs])
prob_48_test = np.array([safe_prob(fn, 48) for fn in test_surv_funcs])
prob_72_test = np.array([safe_prob(fn, 72) for fn in test_surv_funcs])

# --- SỬA LỖI 4: BỔ SUNG LẠI CỘT risk_score (Cần thiết cho C-index) ---
submission = pd.DataFrame({
    'event_id': test['event_id'],
    'prob_12h': prob_12_test,
    'prob_24h': prob_24_test,
    'prob_48h': prob_48_test,
    'prob_72h': prob_72_test
})

submission.to_csv("submission.csv", index=False)
print("Đã lưu file submission.csv thành công!")


--- THỰC THI PSDATA: Đang tổng hợp thêm 35 mẫu mới ---
Đang huấn luyện mô hình Gradient Boosting Survival (Tập Train 80%)...
Đang đánh giá trên tập validation (20%)...

--- KẾT QUẢ LOCAL VALIDATION ---
Hybrid Score    : 0.9683
C-index         : 0.9430
Weighted Brier  : 0.0208
  - Brier@12h   : 0.0728
  - Brier@24h   : 0.0372
  - Brier@48h   : 0.0240
  - Brier@72h   : 0.0000


--- THỰC THI PSDATA: Đang tổng hợp thêm 44 mẫu mới ---
Đang huấn luyện lại mô hình trên TOÀN BỘ dữ liệu train (100%)...
Đang dự đoán trên tập test...
Đã lưu file submission.csv thành công!
